# Unlearning vs. Membership Inference: all experiments

Runs every target model of Table 1 (1a ... 3d) with every unlearning method (NegGrad, SCRUB, SFTC) and fills the paper table `tab:utility_mia_all`.

**What happens for each target model**
1. The data splits used to train the saved models are rebuilt (seed 42, same split functions as `train_target_and_perform_MIA_attack.py`).
2. The saved models are loaded from `models/`: target, retrained-from-scratch baseline, and the shadow-model attack (MIA-1). Nothing is retrained.
3. Each method unlearns the target for `EPOCHS` epochs. After **every** epoch it measures test accuracy and the forget/retain AUC of
   - **MIA-1**: shadow-model + RandomForest attack (saved `.jolib` model)
   - **MIA-2**: loss-threshold attack
   - **MIA-3**: confidence-based attack (maximum softmax probability, label-free)
4. **Best** = epoch minimising `|MIA-1 forget AUC − Retr.| + |Test Acc − Retr. Test Acc|`; **Final** = last epoch. AUC is reported as max(AUC, 1−AUC).

**Reproducibility**
- Fixed seeds, deterministic cuDNN/cuBLAS, every run reseeded at its start, so results don't depend on run order.
- Each target model runs in its own subprocess (clean state, and memory is freed between experiments).
- Results are cached in `results/<exp>/<method>.json` together with their config and software versions. Re-running the notebook skips finished runs, so it **resumes after an interruption**. Changing a learning rate (or epochs/seed) re-runs only that run. To force a re-run, delete its JSON file.

**Run it without keeping the notebook open** (from the project folder):
```
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 run_all_experiments.ipynb
```
Outputs: `results/summary.csv`, `results/table_utility_mia_all.tex`, and per-epoch curves in `results/<exp>/<method>_epochs.csv`.

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # deterministic cuBLAS; set before CUDA starts

import json
import pandas as pd
import experiment_runner as er

env = er.environment_info()
print(json.dumps(env, indent=2))
if env["uncommitted_python_changes"]:
    print("Note: there are uncommitted .py changes; commit them so results can be tied to a git commit.")

/home/theo/miniconda3/envs/unlearn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "python": "3.11.16",
  "torch": "2.0.1+cu117",
  "cuda": "11.7",
  "cudnn": 8500,
  "gpu": "NVIDIA GeForce RTX 3050 Laptop GPU",
  "numpy": "1.23.2",
  "pandas": "1.5.3",
  "sklearn": "1.5.1",
  "git_commit": "1594d05e",
  "uncommitted_python_changes": true,
  "runner_version": 1
}
Note: there are uncommitted .py changes; commit them so results can be tied to a git commit.


## Configuration

`LEARNING_RATES`: one unlearning learning rate per target model and method, taken from the paper figure filenames. Set a value to `None` to skip that run (it shows as `--` in the table). Changing a value re-runs only that run.

In [2]:
SEED = 42
EPOCHS = 30          # unlearning epochs
BATCH_SIZE = 32
MOMENTUM = 0.9       # SGD momentum, as in main.py
RESULTS_DIR = "results"

EXPERIMENTS_TO_RUN = er.ALL_EXPERIMENTS   # e.g. ["1d", "2d", "3d"] to run a subset
METHODS = er.METHODS                       # ["neg_grad", "scrub", "sftc"]

LEARNING_RATES = {   # from the paper figure filenames (fig/exp*/<exp>_<method>_lr_<lr>.pdf)
    "1a": {"neg_grad": 2e-4,    "scrub": 1e-4,    "sftc": 1e-4},
    "1b": {"neg_grad": 2e-4,    "scrub": 2e-4,    "sftc": 1e-3},
    "1c": {"neg_grad": 5e-5,    "scrub": 1e-5,    "sftc": 3e-3},   # SCRUB: figures also had 3e-4
    "1d": {"neg_grad": 1e-5,    "scrub": 3e-4,    "sftc": 1e-2},
    "2a": {"neg_grad": 1.5e-6,  "scrub": 1e-5,    "sftc": 9e-4},
    "2b": {"neg_grad": 6e-5,    "scrub": 7.5e-5,  "sftc": 9e-6},
    "2c": {"neg_grad": 1e-5,    "scrub": 5e-4,    "sftc": 1e-2},
    "2d": {"neg_grad": 6e-6,    "scrub": 1e-4,    "sftc": 1e-2},
    "3a": {"neg_grad": 1e-5,    "scrub": 1e-5,    "sftc": 1e-2},
    "3b": {"neg_grad": 1e-5,    "scrub": 1e-5,    "sftc": 1e-3},
    "3c": {"neg_grad": 1e-6,    "scrub": 1e-5,    "sftc": 1e-3},
    "3d": {"neg_grad": 1e-6,    "scrub": 1e-7,    "sftc": 1e-4},
}

missing = [(e, er.METHOD_NAMES[m]) for e in EXPERIMENTS_TO_RUN for m in METHODS if LEARNING_RATES[e][m] is None]
print(f"{len(missing)} runs have no learning rate yet and will be skipped:")
print(", ".join(f"{e} {m}" for e, m in missing) or "none")

0 runs have no learning rate yet and will be skipped:
none


## Run all experiments
One line per epoch is printed. Finished runs are loaded from `results/` instead of re-run.

In [3]:
failed = er.run_all(EXPERIMENTS_TO_RUN, METHODS, LEARNING_RATES, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    momentum=MOMENTUM, seed=SEED, results_dir=RESULTS_DIR)

Files already downloaded and verified
[1a] CIFAR-10 | TargetModel_1a | retain 12800, forget 3200, test 4000, MIA sample 3200 | loaded in 16s
  Original : test acc 0.489, forget acc 1.000, retain acc 1.000, MIA-1 forget AUC 0.848
  Retrained: test acc 0.478, forget acc 0.478, retain acc 1.000, MIA-1 forget AUC 0.501
  [1a NegGrad lr=0.0002] epoch  1/30 | test 0.489 | forget AUC MIA-1 0.847 MIA-2 0.855 MIA-3 0.806 | 1s
  [1a NegGrad lr=0.0002] epoch  2/30 | test 0.490 | forget AUC MIA-1 0.848 MIA-2 0.855 MIA-3 0.806 | 2s
  [1a NegGrad lr=0.0002] epoch  3/30 | test 0.489 | forget AUC MIA-1 0.849 MIA-2 0.855 MIA-3 0.806 | 3s
  [1a NegGrad lr=0.0002] epoch  4/30 | test 0.489 | forget AUC MIA-1 0.848 MIA-2 0.855 MIA-3 0.806 | 4s
  [1a NegGrad lr=0.0002] epoch  5/30 | test 0.489 | forget AUC MIA-1 0.850 MIA-2 0.855 MIA-3 0.806 | 5s
  [1a NegGrad lr=0.0002] epoch  6/30 | test 0.489 | forget AUC MIA-1 0.851 MIA-2 0.855 MIA-3 0.806 | 6s
  [1a NegGrad lr=0.0002] epoch  7/30 | test 0.489 | forget 

## Results

In [4]:
results = er.load_results(RESULTS_DIR)
summary = er.summary_dataframe(results)
summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
summary.round(3)

,exp,dataset,method,lr,best_epoch,diverged_at_epoch,test_acc_start,forget_mia1_auc_start,forget_mia2_auc_start,forget_mia3_auc_start,...,forget_mia2_auc_retrain,forget_mia3_auc_retrain,test_acc_best,forget_mia1_auc_best,forget_mia2_auc_best,forget_mia3_auc_best,test_acc_final,forget_mia1_auc_final,forget_mia2_auc_final,forget_mia3_auc_final
0,1a,CIFAR-10,NegGrad,0.000,20,21.0,0.489,0.848,0.855,0.806,...,0.505,0.502,0.479,0.809,0.845,0.780,NaN,NaN,NaN,NaN
1,1a,CIFAR-10,SCRUB,0.000,1,2.0,0.489,0.848,0.855,0.806,...,0.505,0.502,0.489,0.847,0.855,0.806,NaN,NaN,NaN,NaN
2,1a,CIFAR-10,SFTC,0.000,30,NaN,0.489,0.848,0.855,0.806,...,0.505,0.502,0.478,0.607,0.796,0.690,0.478,0.607,0.796,0.690
3,1b,MuFac,NegGrad,0.000,12,13.0,0.336,0.911,0.943,0.895,...,0.523,0.503,0.189,0.500,0.504,0.500,NaN,NaN,NaN,NaN
4,1b,MuFac,SCRUB,0.000,13,NaN,0.336,0.911,0.943,0.895,...,0.523,0.503,0.276,0.525,0.506,0.504,0.226,0.503,0.509,0.501
5,1b,MuFac,SFTC,0.001,22,NaN,0.336,0.911,0.943,0.895,...,0.523,0.503,0.324,0.503,0.673,0.514,0.320,0.545,0.740,0.541
6,1c,Purchase-100,NegGrad,0.000,12,NaN,0.800,0.778,0.717,0.710,...,0.502,0.503,0.789,0.749,0.705,0.693,0.027,0.500,0.503,0.500
7,1c,Purchase-100,SCRUB,0.000,17,NaN,0.800,0.778,0.717,0.710,...,0.502,0.503,0.799,0.778,0.717,0.710,0.800,0.778,0.717,0.710
8,1c,Purchase-100,SFTC,0.003,26,NaN,0.800,0.778,0.717,0.710,...,0.502,0.503,0.675,0.572,0.649,0.586,0.667,0.568,0.641,0.574
9,1d,Texas-100,NegGrad,0.000,3,NaN,0.578,0.573,0.683,0.601,...,0.504,0.507,0.578,0.571,0.683,0.601,0.577,0.574,0.683,0.600


## LaTeX table
Written to `results/table_utility_mia_all.tex`; include it with `\input{...}`. Final/Best test accuracies below 25% of the retrained model's are bolded (model collapse).

In [5]:
tex = er.make_latex_table(results)
with open(os.path.join(RESULTS_DIR, "table_utility_mia_all.tex"), "w") as f:
    f.write(tex)
print(tex)

% Requires: \usepackage{booktabs,multirow,graphicx,rotating}
% Generated by experiment_runner.make_latex_table. Rows: Dataset -> Target model ID (Table 1) -> method.
% Start = original model, Retr. = retrained from scratch on the retain set (merged cells).
% Best epoch = argmin |MIA-1 forget AUC - Retr.| + |Test Acc - Retr.|. '--' = not run yet.

\providecommand{\startcell}[1]{\multirow{3}{*}{#1}}

\begin{sidewaystable}
\centering
\caption{Test accuracy and forget-set MIA AUC for all 12 target models (Table~1) and all three
unlearning methods: the original model before unlearning (Start), the model retrained from scratch
on the retain set (Retr.), the unlearned model at the epoch closest to the retrained model (Best;
epoch in parentheses), and after the last unlearning epoch (Final). Start and Retr. do not depend on
the unlearning method and are therefore given once per target model. MIA-2 and MIA-3 are evaluated at
the same epochs as MIA-1. AUC is reported as $\max(\mathrm{AUC}, 1-\ma

## Optional: per-epoch curves for one run

In [6]:
import matplotlib.pyplot as plt

EXP, METHOD = "3c", "sftc"   # pick any finished run
path = os.path.join(RESULTS_DIR, EXP, f"{METHOD}_epochs.csv")
if os.path.exists(path):
    h, r = pd.read_csv(path), results[EXP][METHOD]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["epoch"], h["test_acc"], label="Test acc")
    axes[0].axhline(r["retrain"]["test_acc"], ls="--", c="gray", label="Retrained")
    for a, c in zip(er.ATTACKS, ["C0", "C1", "C2"]):
        axes[1].plot(h["epoch"], h[f"forget_{a}_auc"], c=c, label=a.upper().replace("MIA", "MIA-"))
        axes[1].axhline(r["retrain"][f"forget_{a}_auc"], ls="--", c=c)
    for ax, t in zip(axes, ["Test accuracy", "Forget-set MIA AUC (dashed: retrained)"]):
        ax.axvline(r["best_epoch"], c="k", ls=":", label="Best epoch")
        ax.set_title(t); ax.set_xlabel("Unlearning epoch"); ax.legend(); ax.grid(alpha=.3)
    fig.suptitle(f"{EXP} {er.METHOD_NAMES[METHOD]} (lr={r['config']['lr']:g})")
    plt.show()
else:
    print(f"No results yet for {EXP} {METHOD}")

<Figure size 1200x400 with 2 Axes>